# 03 — Construire la base pour l’analyse

**Suite du notebook 02 :** nous appliquons maintenant les choix de préparation, sans modifier les sources. Une ligne représente **une forme PokéAPI**. Toutes les formes restent présentes, même lorsqu’elles partagent leurs statistiques. Les talents et conditions d’évolution sont exclus.

Les fonctions se trouvent dans `src/preparation.py` et les différences de noms dans `config/preparation.json`. Ce notebook se lance indépendamment des précédents, depuis le dossier du projet ou `notebook/`.

La table commune du notebook 01 est maintenant l’entrée de la préparation. Les deux collectes sont rechargées et comparées ; on construit ensuite la base avec chacune pour vérifier que l’enrichissement donne aussi le même résultat. Le choix `source_pokeapi` dans `config/preparation.json` détermine quelle version sera exportée (CSV par défaut).

In [1]:
from pathlib import Path
import sys
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == "notebook":
    project_dir = project_dir.parent
assert (project_dir / "src" / "preparation.py").exists(), "Ouvrir depuis le projet ou notebook/."
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

import pandas as pd
from src.collect import (collect_sources, load_sources, check_sources,
                         pokeapi_from_csv, collect_pokeapi_api, compare_pokeapi)
from src import preparation as prep

collect_sources()  # Vérifie les empreintes de l’instantané partagé.
data = load_sources()
source_checks = check_sources(data)

pokeapi_csv = pokeapi_from_csv(data)
pokeapi_api = collect_pokeapi_api()  # Réutilise le cache du notebook 01.
comparaison = compare_pokeapi(pokeapi_csv, pokeapi_api)
display(comparaison["resume"])
display(comparaison["ecarts"].head(20))
assert comparaison["identiques"], "Examiner les écarts CSV/API dans le notebook 01 avant la préparation."


C:\Users\Ariel\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Conservé et vérifié : pokemon_species.csv
Conservé et vérifié : pokemon.csv
Conservé et vérifié : pokemon_stats.csv
Conservé et vérifié : stats.csv
Conservé et vérifié : pokemon_species_names.csv
Conservé et vérifié : pokemon_types.csv
Conservé et vérifié : type_names.csv
Conservé et vérifié : showdown_pokedex.json
Conservé et vérifié : pokemon_forms.csv
Conservé et vérifié : pokemon_form_types.csv
Conservé et vérifié : version_groups.csv
Contrôles terminés : 1579 formes conservées ; aucun filtre ni regroupement.


Catalogue API : 1579 formes. Les réponses en cache seront réutilisées.


Formes lues : 100/1579


Formes lues : 200/1579


Formes lues : 300/1579


... (11 outputs left out)

Formes lues : 1500/1579


Formes lues : 1579/1579


,nombre
Formes CSV,1579
Formes API,1579
Formes communes,1579
Formes seulement CSV,0
Formes seulement API,0
Colonnes comparées hors identifiant,26
Cellules différentes sur les formes communes,0


,form_id,champ,csv,api


## 1. Quelles variables construit-on ?

| Colonnes | Définition et source |
|---|---|
| `form_id`, `pokemon_id`, `species_id` | Identifiants de la forme (clé unique), de la variété et de l’espèce PokéAPI. |
| `nom_fr`, `forme` | Nom français **de l’espèce**, puis identifiant complet de la forme. Le français des noms de formes n’est pas présent dans les sources collectées : aucune traduction n’est inventée. |
| `generation` | Génération du groupe de jeux d’introduction de **la forme**, indiqué par PokéAPI. |
| `pv`, `attaque`, `defense`, `attaque_speciale`, `defense_speciale`, `vitesse` | Six statistiques de base de la variété à laquelle la forme appartient. |
| `bst` | Somme des six statistiques, pour étudier leur niveau global. |
| `type_1`, `type_2` | Types en français : types propres à la forme lorsqu’ils sont renseignés, sinon types de sa variété. Le second type est vide pour un mono-type. |
| `tag` | Catégories positives identifiées : Légendaire, Fabuleux, Paradoxe, Ultra-Chimère. Plusieurs catégories peuvent coexister, séparées par `; `. |
| `tier` | Champ `tier` de l’instantané Showdown, directement renseigné ou obtenu par une référence documentée. |
| `is_default_form`, `is_battle_only`, `is_mega` | Indicateurs PokéAPI conservés pour décrire la population et permettre des analyses ultérieures à périmètre explicite. |

**Dates et valeurs ne racontent pas la même chose :** la génération est une date d’introduction, tandis que les statistiques sont celles de l’instantané actuel. Nous n’avons pas reconstitué leurs valeurs historiques.

**Rapprochement Showdown :** noms normalisés, noms déclarés dans `baseForme` / `cosmeticFormes`, puis correspondances de libellés explicites dans la configuration. Les statistiques ne servent jamais à choisir une fiche. Le numéro d’espèce, les statistiques et les types sont contrôlés ensuite. Les 63 formes de Charmilly gardent leur ligne : Showdown ne distingue pas leurs sucreries. Les sept couleurs de Météno météore gardent aussi leur ligne malgré une représentation Showdown commune.

In [2]:
showdown = data["showdown_pokedex"]
base_csv, audit_csv = prep.prepare_database(pokeapi_csv, showdown, source="csv")
base_api, audit_api = prep.prepare_database(pokeapi_api, showdown, source="api")
pd.testing.assert_frame_equal(base_csv, base_api)

source = prep.read_preparation_config()["source_pokeapi"]
assert source in ["csv", "api"], "Choisir csv ou api dans config/preparation.json."
if source == "csv":
    pokeapi, base, audit = pokeapi_csv, base_csv, audit_csv
else:
    pokeapi, base, audit = pokeapi_api, base_api, audit_api
print("Bases finales identiques avec les deux collectes. Source retenue :", source)
display(base.head())

Bases finales identiques avec les deux collectes. Source retenue : csv


,form_id,pokemon_id,species_id,nom_fr,forme,generation,pv,attaque,defense,attaque_speciale,defense_speciale,vitesse,type_1,type_2,is_default_form,is_battle_only,is_mega,bst,tag,tier
0,1,1,1,Bulbizarre,bulbasaur,1,45,49,49,65,65,45,Plante,Poison,1,0,0,318,Non étiqueté,LC
1,2,2,2,Herbizarre,ivysaur,1,60,62,63,80,80,60,Plante,Poison,1,0,0,405,Non étiqueté,NFE
2,3,3,3,Florizarre,venusaur,1,80,82,83,100,100,80,Plante,Poison,1,0,0,525,Non étiqueté,ZU
3,4,4,4,Salamèche,charmander,1,39,52,43,60,50,65,Feu,<NA>,1,0,0,309,Non étiqueté,LC
4,5,5,5,Reptincel,charmeleon,1,58,64,58,80,65,80,Feu,<NA>,1,0,0,405,Non étiqueté,NFE


## 2. La fusion conserve-t-elle nos observations ?

La fonction vérifie l’unicité et la conservation de **tous les identifiants de forme**, les rattachements et les variables essentielles. L’audit conserve la fiche choisie, la méthode, les sources des enrichissements et les accords/désaccords. Une divergence de statistiques, de types ou de numéro d’espèce est signalée : elle ne corrige pas les valeurs PokéAPI et bloque l’export tant qu’elle n’a pas été examinée. Une forme sans correspondance reste conservée.

Dans `reference_01`, les **1 579 formes** sont conservées ; les **1 518 rapprochements** donnent des statistiques et types concordants. Ce total désigne des lignes PokéAPI comparées, pas autant de fiches Showdown distinctes. Les 61 lignes restantes ne sont pas vérifiées par Showdown : elles ne constituent ni des accords ni des désaccords.

Les types Showdown sont traduits en identifiants de type avec le dictionnaire de `config/preparation.json`, issu des noms anglais de PokéAPI. Cela permet de comparer les types sans modifier les libellés français de la table commune. `source_collecte` indique CSV/API et `source_generation` conserve l’identifiant du groupe de jeux d’introduction.

In [3]:
display(prep.summarize_database(base, audit))
display(audit["methode_jointure"].value_counts().rename("formes").to_frame())

# Ce tableau resterait visible si une nouvelle source introduisait une divergence.
divergences = audit.loc[
    audit["stats_concordantes"].eq(False) | audit["types_concordants"].eq(False),
    ["forme", "showdown_id", "stats_divergentes", "types_pokeapi", "types_showdown"]
]
display(divergences)

,nombre
Formes conservées,1579
Espèces représentées,1025
Sans nom français,0
Sans génération,0
Sans correspondance Showdown,61
Correspondances à une représentation cosmétique,70
Statistiques comparées,1518
Divergences de statistiques,0
Types comparés,1518
Divergences de types,0


,formes
methode_jointure,
nom_direct,1307
alias_declare_showdown,99
representation_cosmetique,70
non_apparie,61
alias_configure,42


,forme,showdown_id,stats_divergentes,types_pokeapi,types_showdown


## 3. Comprendre les absences avant de les traiter

**PokéAPI.** Aucun nom français d’espèce, génération, premier type ou statistique ne manque ici. Les **777 seconds types vides** correspondent aux mono-types : ils restent vides. `arceus-unknown` porte la valeur littérale `???`, distincte d’une donnée manquante ; cette entrée reste dans l’inventaire et devra être isolée explicitement pour une analyse des types jouables.

**Showdown.** Les **61 tiers absents** concernent des correspondances non établies par nos règles :

- 43 formes de Papilord, Lépidonille et Pérégrain, dont les distinctions ne sont pas explicitées par les alias utilisés ;
- 6 formes mâles/femelles de Viskuse, Moyade et Némélios ;
- 8 modes de déplacement de Koraidon et Miraidon ;
- 2 variantes de Zygarde et 1 de Rocabot, non rapprochées ici ;
- `arceus-unknown`.

**Traitement retenu :** conserver ces lignes et laisser leur tier vide. Cela ne prouve pas que les 61 formes sont absentes de Showdown : certaines y sont décrites à un niveau moins détaillé, et d’autres demandent encore une justification de correspondance. On pourra enrichir la configuration plus tard, sans modifier la population de la base. Aucun tier `Illegal` n’est créé pour combler une absence.

Parmi les tiers disponibles, **1 372 sont explicites** dans la fiche choisie et **146 proviennent de références**. Les références de combat ont priorité sur `baseSpecies`, avec les cas Mimikyu/Totem prévus par le [moteur Showdown](https://github.com/smogon/pokemon-showdown/blob/master/sim/dex-species.ts). La règle et la fiche d’origine figurent dans l’audit. Il s’agit d’une lecture documentée du JSON sauvegardé, pas d’une exécution complète du simulateur.

In [4]:
sans_tier = base.loc[base["tier"].isna()]
display(sans_tier.groupby("nom_fr", sort=True).agg(
    formes=("form_id", "size"), exemple=("forme", "first")
))
display(audit["regle_tier"].value_counts().rename("formes").to_frame())

# Toutes les lignes restent consultables, sans allonger l’affichage du notebook.
formes_a_revoir = audit.loc[audit["showdown_id"].isna()]

,formes,exemple
nom_fr,,
Arceus,1,arceus-unknown
Koraidon,4,koraidon-limited-build
Lépidonille,20,scatterbug-icy-snow
Miraidon,4,miraidon-low-power-mode
Moyade,2,jellicent-male
Némélios,2,pyroar-male
Papilord,3,mothim-plant
Pérégrain,20,spewpa-icy-snow
Rocabot,1,rockruff-own-tempo


,formes
regle_tier,
explicite,1372
reference_baseSpecies,123
sans_correspondance,61
reference_combat,22
reference_mimikyu,1


## 4. Que signifie le tag ?

Les indicateurs `is_legendary` et `is_mythical` viennent de PokéAPI. Pour les Paradoxes, on combine les preuves positives des tags Showdown et de la catégorie anglaise `Paradox Pokémon` de PokéAPI. Cette dernière couvre **22 espèces**, dont Koraidon, Miraidon et les quatre espèces récentes non étiquetées Paradox dans ce JSON Showdown. Le tag Ultra-Chimère vient des **11 espèces** explicitement étiquetées `Ultra Beast` par Showdown.

Ces catégories sont attribuées **au niveau de l’espèce**, puis transmises à ses formes, même sans correspondance de forme Showdown. Les provenances sont dans `source_tag`. Les autres tags Showdown, par exemple `True Past`, ne sont pas des catégories biologiques retenues pour la comparaison ; ceux de la fiche rapprochée restent consultables dans l’audit.

**`Non étiqueté` signifie qu’aucune de ces catégories n’a été identifiée avec ces critères, pas que la source garantit un Pokémon « ordinaire ».** Une absence de tag ne devient donc pas une fausse certitude. Koraidon et Miraidon cumulent Légendaire et Paradoxe : les catégories ne forment pas une partition. Les effectifs d’espèces évitent de confondre couverture des catégories et multiplication des formes.

In [5]:
display(base.groupby("tag").agg(
    formes=("form_id", "size"), especes=("species_id", "nunique")
))

# Cas illustrant les choix de représentation, sans les programmer un par un.
exemples = ["charizard", "charizard-gmax", "arceus-fairy", "arceus-unknown",
            "silvally-fairy", "minior-red-meteor", "minior-red", "koraidon-apex-build"]
display(base.loc[base["forme"].isin(exemples),
    ["form_id", "nom_fr", "forme", "generation", "type_1", "type_2", "bst", "tag", "tier"]])

,formes,especes
tag,,
Fabuleux,60,23
Légendaire,128,69
Légendaire; Paradoxe,10,2
Non étiqueté,1350,900
Paradoxe,20,20
Ultra-Chimère,11,11


,form_id,nom_fr,forme,generation,type_1,type_2,bst,tag,tier
5,6,Dracaufeu,charizard,1,Feu,Vol,534,Non étiqueté,ZU
773,774,Météno,minior-red-meteor,7,Roche,Vol,440,Non étiqueté,ZU
1006,1007,Koraidon,koraidon-apex-build,9,Combat,Dragon,670,Légendaire; Paradoxe,Uber
1081,10057,Arceus,arceus-unknown,4,???,<NA>,720,Fabuleux,NaN
1109,10085,Arceus,arceus-fairy,6,Fée,<NA>,720,Fabuleux,Uber
1272,10248,Silvallié,silvally-fairy,7,Fée,<NA>,570,Légendaire,Illegal
1279,10255,Météno,minior-red,7,Roche,Vol,500,Non étiqueté,ZU
1389,10365,Dracaufeu,charizard-gmax,8,Feu,Vol,534,Non étiqueté,Illegal


## 5. Export et limites pour l’analyse

Deux CSV régénérables sont écrits dans `data/processed/reference_01/` : `pokemon_forms.csv` pour l’analyse et `audit_preparation.csv` pour retrouver chaque choix grâce à `form_id`. Un troisième fichier, `provenance.json`, conserve la source retenue, les instantanés et les empreintes des sources de référence, du code, de la configuration et des deux CSV exportés. Ce dossier est ignoré par Git ; chacun peut le reconstruire avec le même instantané et la même configuration. Aucun fichier brut n’est modifié.

La base est exhaustive **par rapport aux formes de l’instantané PokéAPI**, sans garantir que la source inventorie parfaitement toute la licence. Les formes cosmétiques, de combat, Gigamax et historiques restent distinctes. Par conséquent, une espèce riche en formes pèse davantage dans une moyenne par ligne : on devra comparer les résultats par forme avec des résultats au niveau variété ou espèce. Les formes d’une espèce ne sont pas des observations indépendantes.

Le tier dépend du contexte compétitif et de la date du JSON ; ce n’est ni un score numérique de puissance ni le tier historique à l’introduction de la forme. Une valeur `Illegal` présente dans la source reste une valeur renseignée. Toute analyse du tier devra annoncer le format étudié et le périmètre retenu, ainsi que les formes sans tier. Les talents, attaques et conditions d’évolution ne sont pas analysés : le BST seul n’explique pas la performance en combat.

In [6]:
prep.validate_database(base, audit, pokeapi)
output_dir = prep.export_database(base, audit, pokeapi)
print("Base, audit et provenance enregistrés dans :", output_dir)

Base, audit et provenance enregistrés dans : C:\Users\Ariel\OneDrive - ENSEA\Documents\Python pour la DS\Projet\Projet-python-data-science\data\processed\reference_01
